# 1. Příprava prostředí a načtení dat
V tomto kroku importujeme potřebné knihovny pro práci s daty a strojové učení. Následně načteme dataset s geocaching daty ze souboru `gsak.csv`.

In [4]:
import pandas as pd
import re
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('gsak.csv', header=None, encoding='utf-8')
df.columns = ['Code', 'Name', 'Type', 'Date', 'Size', 'Diff', 'Terr', 'Lat', 'Lon', 'Country', 'State', 'County', 'Elev']
df

,Code,Name,Type,Date,Size,Diff,Terr,Lat,Lon,Country,State,County,Elev
0,GC1RB93,Motorkarska cache I. / Bikers cache I.,Unknown Cache,19.06.2009,Small,"3,0","2,0",N50° 07.700,E14° 25.500,Czechia,Hlavní mesto Praha,Hlavni mesto Praha,302.0
1,GC1Y1QM,URBAN LEGENDS Vol. 1,Unknown Cache,31.08.2009,Small,"2,5","3,0",N50° 06.760,E14° 27.030,Czechia,Hlavní mesto Praha,Hlavni mesto Praha,177.0
2,GC1ZYND,GeoPLB#2 - Pribeh pacienta c. 916,Unknown Cache,17.10.2009,Regular,"2,5","1,5",N50° 08.000,E14° 25.437,Czechia,Hlavní mesto Praha,Hlavni mesto Praha,287.0
3,GC12PQ2,Holesovicky industrial,Multicache,07.05.2007,Small,"1,5","2,0",N50° 06.328,E14° 27.005,Czechia,Hlavní mesto Praha,Hlavni mesto Praha,195.0
4,GC2J3AT,Kvetnove povstani / May Uprising,Multicache,11.11.2010,Micro,"1,5","2,5",N50° 06.140,E14° 27.335,Czechia,Hlavní mesto Praha,Hlavni mesto Praha,187.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2542,GC90JP9,Challenge Cache Series - The Matrix Cacher,Unknown Cache,26.09.2020,Regular,"3,5","2,5",S34° 47.505,E148° 49.094,Australia,New South Wales,Yass Valley,0.0
2543,GC8ZMFC,Challenge Cache Series - Cache Types,Unknown Cache,16.08.2020,Small,"3,0","1,5",S34° 48.532,E148° 48.094,Australia,New South Wales,Yass Valley,0.0
2544,GC8ZXJ5,Challenge Cache Series - Hiking Attribute,Unknown Cache,31.08.2020,Small,"4,0","1,5",S34° 48.687,E148° 48.248,Australia,New South Wales,Yass Valley,0.0
2545,GC5ARPA,John French VC Memorial Rest Area,Multicache,12.08.2014,Small,"2,5","1,5",S34° 47.222,E149° 38.417,Australia,New South Wales,Goulburn Mulwaree,0.0


# 2. Zpracování dat a Feature Engineering
Aby mohl model data pochopit, musíme je transformovat.
* Převádíme GPS souřadnice z textového formátu na desetinná čísla.
* Upravujeme textové zápisy obtížnosti a terénu.
* Vytváříme nové atributy: `Is_Drive_In` (snadno dostupné keše s nízkým terénem) a `Is_Puzzle` (mysterky, obecně těžší keše).

In [5]:
def parse_coords(s):
    match = re.search(r'([NSEW])(\d+)\u00b0\s*(\d+\.\d+)', str(s))
    if match:
        d, deg, min = match.groups()
        val = float(deg) + float(min) / 60.0
        return val if d in ['N', 'E'] else -val
    return None

df['Lat_N'] = df['Lat'].apply(parse_coords)
df['Lon_N'] = df['Lon'].apply(parse_coords)
df['Diff_N'] = df['Diff'].astype(str).str.replace(',', '.').str.extract(r'(\d+\.?\d*)')[0].astype(float)
df['Terr_N'] = df['Terr'].astype(str).str.replace(',', '.').str.extract(r'(\d+\.?\d*)')[0].astype(float)

def get_drive_in_index(row):
    if row['Terr_N'] <= 1.5 and row['Size'] in ['Micro', 'Small', 'Other']:
        return 1
    return 0
df['Is_Drive_In'] = df.apply(get_drive_in_index, axis=1)

def get_puzzle_index(row):
    name_lower = str(row['Name']).lower()
    if row['Type'] == 'Unknown Cache' in name_lower:
        return 1
    return 0
df['Is_Puzzle'] = df.apply(get_puzzle_index, axis=1)

features = ['Type', 'Size', 'Terr_N', 'Lat_N', 'Lon_N', 'Is_Drive_In', 'Is_Puzzle']
df_ml = df.dropna(subset=features + ['Diff_N']).copy()

# 3. Encoding textových dat
Model strojového učení nedokáže pracovat s textem (např. "Micro", "Traditional Cache"). Proto tyto hodnoty pomocí `LabelEncoder` převedeme na číselný typ.

In [6]:
le_type = LabelEncoder()
df_ml['Type_E'] = le_type.fit_transform(df_ml['Type'])

le_size = LabelEncoder()
df_ml['Size_E'] = le_size.fit_transform(df_ml['Size'])

X = df_ml[['Type_E', 'Size_E', 'Terr_N', 'Lat_N', 'Lon_N', 'Is_Drive_In', 'Is_Puzzle']]
y = df_ml['Diff_N'].astype(str)

# 4. Příprava pro trénování (Split & Scaling)
Data rozdělíme v poměru 80:20. Na 80 % dat se model bude učit a na zbylých 20 % model následně otestujeme, abychom zaručili, že se data nenaučil pouze nazpaměť. Následně data škálujeme pomocí `StandardScaler`, abychom sjednotili jejich rozsahy.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Model Random Forest a jeho vyhodnocení
Pomocí algoritmu RandomForestClassifier, natrénujeme model na škálovaných datech. Poté provedeme predikci na testovací sadě a vypíšeme metriky úspěšnosti.

In [11]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print("--- VÝSLEDKY MODELU NA TESTOVACÍCH DATECH ---")
print(f"Celková přesnost (přesná shoda): {accuracy_score(y_test, y_pred)*100:.2f} %")
y_test_num = y_test.astype(float).values
y_pred_num = y_pred.astype(float)
errors = np.abs(y_test_num - y_pred_num)

print(f"Shoda s tolerancí ±0.5 obtížnosti: {(errors <= 0.5).mean()*100:.1f} %")
print(f"Shoda s tolerancí ±1.0 obtížnosti: {(errors <= 1.0).mean()*100:.1f} %")

--- VÝSLEDKY MODELU NA TESTOVACÍCH DATECH ---
Celková přesnost (přesná shoda): 34.51 %
Shoda s tolerancí ±0.5 obtížnosti: 59.2 %
Shoda s tolerancí ±1.0 obtížnosti: 75.3 %


# 6. Export modelu
Všechny natrénované komponenty (samotný model, scaler i encodery) uložíme pomocí knihovny `joblib` do souborů `.pkl`.

In [12]:
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le_type, 'le_type.pkl')
joblib.dump(le_size, 'le_size.pkl')

['le_size.pkl']